# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [24]:
# Importation des packages nécessaires

import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [25]:
# Chargement des données
df_mapping_initial = pd.read_csv("../data_finale/mapping_fbref_tm.csv", encoding='latin1')
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

df_tm_initial = pd.merge(df_players, df_valuations, on="player_id", how="inner")


Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [26]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [27]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm)

[1] Nom exact (mapping)     : 17121 | restants : 833
[2] Fuzzy nom (mapping)     :    55 | restants : 778


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [28]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')

In [29]:
df_final

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,market_value_in_eur_x,highest_market_value_in_eur,date,market_value_in_eur_y,current_club_name_y,current_club_id_y,player_club_domestic_competition_id,tm_join_key,tm_join_key_full,tm_dob_key
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,9000000.0,18000000.0,2021-12-23,12000000.0,Arsenal FC,1041.0,FR1,ainsley maitland niles,ainsley maitland niles,1997
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,5000000.0,70000000.0,2021-12-23,20000000.0,Arsenal FC,34911.0,SA1,alexandre lacazette,alexandre lacazette,1991
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,8000000.0,35000000.0,2021-12-23,16000000.0,Arsenal FC,931.0,GB1,bernd leno,bernd leno,1992
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,130000000.0,150000000.0,2021-12-23,65000000.0,Arsenal FC,11.0,GB1,bukayo saka,bukayo saka,2001
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,4000000.0,18000000.0,2021-12-23,12000000.0,Arsenal FC,11.0,GB1,calum chambers,calum chambers,1995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17171,ITA-Serie A,2425,Parma,Mathias Løvik,NOR,"DF,MF",20,2003.0,6,0,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17172,ITA-Serie A,2425,Venezia,Mirko Marić,CRO,FW,29,1995.0,10,4,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17173,ITA-Serie A,2526,Genoa,Albert Grønbaek,DEN,MF,24-346,2001.0,4,1,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17174,ITA-Serie A,2526,Parma,Mathias Løvik,NOR,MF,22-149,2003.0,9,5,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
